# Proyecto integrador · Sesión 5 — De los archivos a una base de datos SQL y un tablero
**PY-EAN · Python para Procesamiento, Calidad y Automatización de Datos** · Universidad Ean

**Caso:** la Dirección del fondo de pensiones (ficticio) ya no quiere un Excel por correo. Quiere:

1. Los datos en una **base de datos formal**, con reglas que la propia base haga cumplir.
2. Respuestas con **sentencias SQL**, el idioma que entienden todas las herramientas de la empresa.
3. Un **tablero** que se actualice solo, y saber cómo estamos frente a los **datos reales de Colpensiones**.

| Parte | Qué haces | Tiempo |
|---|---|---|
| 0 | Configuración | 2 min |
| 1 | El pipeline de la semana (ya construido) | 5 min |
| 2 | Datos reales de Colpensiones y del DANE | 10 min |
| 3 | Modelo formal: `CREATE TABLE`, llaves primarias y foráneas | 8 min |
| 4 | Cargar la base y ver cómo se defiende sola | 5 min |
| 5 | **Consultas SQL** (8 ejercicios) | 25 min |
| 6 | Vistas: la capa que alimenta el tablero | 5 min |
| 7 | Tablero | 5 min |

**Cómo trabajar:** Shift + Enter, de arriba hacia abajo. Donde diga `# TU CÓDIGO AQUÍ`, escribe tu solución.
Si Colab se reinicia: **Entorno de ejecución → Ejecutar todas**.

> Los datos del fondo (afiliados, aportes, PQRS) son **ficticios**. Los datos de la Parte 2 son **reales y abiertos**
> (datos.gov.co, licencia CC BY-SA 4.0): conteos agregados, sin datos personales.

## Parte 0 · Configuración

In [ ]:
# ▶️ Ejecuta esta celda primero (Shift + Enter)
import os
import pandas as pd

URL_DATOS = "https://raw.githubusercontent.com/Hackwy402/python-procesamiento-ean/main/datos/"
# Si subiste la carpeta "datos" a Colab (o trabajas en tu PC) se usa la copia local;
# si no, se leen los datos directamente desde GitHub.
for carpeta in ["datos/", "../datos/", "/content/datos/"]:
    if os.path.exists(carpeta + "afiliados.csv"):
        RUTA = carpeta
        break
else:
    RUTA = URL_DATOS

pd.set_option("display.max_columns", 30)
print("Leyendo datos desde:", RUTA)

In [ ]:
import re
import sqlite3
import unicodedata
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("salidas", exist_ok=True)
RUTA_DB = "salidas/bodega_pensiones.db"
print("SQLite", sqlite3.sqlite_version, "· la base quedará en", RUTA_DB)

## Parte 1 · El pipeline de la semana (ya construido)
Estas funciones son las de las sesiones 1 a 4: leer los 9 meses (julio llega con `;`), estandarizar afiliados,
quitar duplicados y calcular días hábiles de respuesta de las PQRS. **Solo ejecútalas**: hoy el foco es SQL.

In [ ]:
CONFIG = {
    "SMMLV": 1_750_905,             # salario mínimo 2026 (dato de referencia del curso)
    "TASA_APORTE": 0.16,            # aporte a pensión = 16 % del IBC
    "DIAS_HABILES_RESPUESTA": 15,   # término general de respuesta a peticiones
    "MESES": range(1, 10),          # enero a septiembre de 2026
    "MESES_MORA": 3,                # 3 o más meses sin aportar = mora
    "SAL": "ean-curso-2026",        # en la vida real: fuera del código
}
FESTIVOS_2026 = ["2026-01-01", "2026-01-12", "2026-03-23", "2026-04-02", "2026-04-03",
                 "2026-05-01", "2026-05-18", "2026-06-08", "2026-06-15", "2026-06-29",
                 "2026-07-20", "2026-08-07", "2026-08-17", "2026-10-12", "2026-11-02",
                 "2026-11-16", "2026-12-08", "2026-12-25"]


def log(mensaje):
    """Bitácora mínima: hora + mensaje."""
    print(f"[{pd.Timestamp.now():%H:%M:%S}] {mensaje}")


def log(mensaje):
    """Bitácora mínima: hora + mensaje."""
    print(f"[{pd.Timestamp.now():%H:%M:%S}] {mensaje}")

In [ ]:
def cargar_curso(ruta):
    """Datos del curso: afiliados, PQRS, los 9 meses de aportes y el catálogo de ciudades."""
    afiliados = pd.read_csv(ruta + "afiliados.csv", dtype={"numero_documento": str, "telefono": str})
    pqrs = pd.read_csv(ruta + "pqrs.csv", parse_dates=["fecha_radicacion", "fecha_respuesta"])
    catalogo = pd.read_excel(ruta + "oficinas.xlsx", sheet_name="ciudades")
    partes = []
    for mes in CONFIG["MESES"]:
        df = pd.read_csv(f"{ruta}aportes/aportes_2026-{mes:02d}.csv", sep=None, engine="python")
        df.columns = df.columns.str.strip().str.lower()        # julio llega en MAYÚSCULA y con ;
        partes.append(df)
    aportes = pd.concat(partes, ignore_index=True)
    log(f"Curso: {len(afiliados):,} afiliados · {len(pqrs):,} PQRS · {len(aportes):,} aportes")
    return afiliados, pqrs, aportes, catalogo

def normalizar_texto(s):
    """minúsculas, sin tildes, sin puntos ni signos, espacios simples."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = re.sub(r"\(.*?\)", "", s.lower())                     # quita aclaraciones: "(amz)"
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


ALIAS_CIUDAD = {"bogota": "bogota d c", "bogota dc": "bogota d c", "b quilla": "barranquilla", "b manga": "bucaramanga",
                "santiago de cali": "cali", "cartagena de indias": "cartagena"}


def a_fecha(serie):
    """Acepta AAAA-MM-DD y DD/MM/AAAA; fechas imposibles → NaT."""
    f = pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")
    f = f.fillna(pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce"))
    return f.where((f >= "1920-01-01") & (f <= pd.Timestamp.today()))

def limpiar_afiliados(df, catalogo):
    """Estandariza afiliados (lo de la Sesión 2)."""
    df = df.drop_duplicates().copy()
    texto = ["tipo_documento", "numero_documento", "nombres", "apellidos", "genero",
             "ciudad", "departamento", "email", "estado"]
    df[texto] = df[texto].apply(lambda col: col.str.strip())
    df["numero_documento"] = df["numero_documento"].str.replace(r"[^0-9]", "", regex=True)
    df["nombres"] = df["nombres"].str.title()
    df["apellidos"] = df["apellidos"].str.title()
    df["genero"] = df["genero"].str.upper().str[0]
    df["estado"] = df["estado"].str.capitalize().replace({"Pensionada": "Pensionado"})
    canon = {normalizar_texto(c): c for c in catalogo["ciudad"]}
    df["ciudad"] = df["ciudad"].map(normalizar_texto).replace(ALIAS_CIUDAD).map(canon).fillna(df["ciudad"])
    df["fecha_nacimiento"] = a_fecha(df["fecha_nacimiento"])
    df["fecha_afiliacion"] = a_fecha(df["fecha_afiliacion"])
    sal = pd.to_numeric(df["salario_base"].astype(str).str.replace(".", "", regex=False), errors="coerce")
    df["salario_base"] = sal.where(sal > 0)
    df["email"] = df["email"].str.lower()
    return df

def limpiar_aportes(df):
    """Quita duplicados exactos y convierte la fecha de pago."""
    df = df.drop_duplicates().copy()
    df["fecha_pago"] = pd.to_datetime(df["fecha_pago"], errors="coerce")
    return df

def dias_habiles_pqrs(pq):
    """Días hábiles de respuesta y si fue a tiempo (solo PQRS cerradas con fechas válidas)."""
    pq = pq.copy()
    validas = ((pq["estado"] == "Cerrado") & pq["fecha_respuesta"].notna()
               & (pq["fecha_respuesta"] >= pq["fecha_radicacion"]))
    inicio = pq.loc[validas, "fecha_radicacion"].values.astype("datetime64[D]")
    fin = pq.loc[validas, "fecha_respuesta"].values.astype("datetime64[D]")
    pq["dias_habiles"] = pd.Series(np.busday_count(inicio, fin, holidays=FESTIVOS_2026),
                                   index=pq.index[validas]).astype("Int64")
    pq["a_tiempo"] = (pq["dias_habiles"] <= CONFIG["DIAS_HABILES_RESPUESTA"]).astype("Int64")
    return pq

In [ ]:
afiliados_raw, pqrs, aportes_raw, catalogo = cargar_curso(RUTA)
afiliados = limpiar_afiliados(afiliados_raw, catalogo)
aportes = limpiar_aportes(aportes_raw)
log(f"Afiliados {len(afiliados_raw):,} → {len(afiliados):,} · aportes {len(aportes_raw):,} → {len(aportes):,}")

## Parte 2 · Datos reales de Colpensiones y del DANE
Cuatro conjuntos de **datos abiertos** de [datos.gov.co](https://www.datos.gov.co), corte 30 oct 2026:

| Archivo | Qué trae | Publica |
|---|---|---|
| `colpensiones_afiliados_municipio.csv` | Afiliados por departamento, municipio y género | Colpensiones |
| `colpensiones_pensionados_municipio.csv` | Pensionados por departamento y municipio | Colpensiones |
| `colpensiones_pensionados_edad.csv` | Pensionados por rango de edad, salario, género y tipo de pensión | Colpensiones |
| `divipola_municipios.csv` | Código oficial de cada municipio, con latitud y longitud | DANE |

Se pueden leer **en vivo** desde la API del portal: `pd.read_csv("https://www.datos.gov.co/resource/xfsi-rqje.csv")`.
Aquí usamos una copia descargada para que todos obtengamos las mismas cifras.

In [ ]:
def cargar_reales(ruta):
    """Datos abiertos reales (datos.gov.co): Colpensiones y DIVIPOLA del DANE."""
    r = ruta + "reales/"
    afil = pd.read_csv(r + "colpensiones_afiliados_municipio.csv")
    pens = pd.read_csv(r + "colpensiones_pensionados_municipio.csv")
    edad = pd.read_csv(r + "colpensiones_pensionados_edad.csv")
    divipola = pd.read_csv(r + "divipola_municipios.csv", dtype=str)
    log(f"Reales: {afil['cantidad'].sum():,} afiliados y {pens['cantidad'].sum():,} pensionados "
        f"en {len(divipola):,} municipios DIVIPOLA")
    return afil, pens, edad, divipola


afil_real, pens_real, edad_real, divipola = cargar_reales(RUTA)
afil_real.head()

In [ ]:
divipola.head(3)

**Ejercicio 1.** Mira `latitud` y `longitud` en DIVIPOLA: vienen como texto con **coma decimal** (`6,246631`).
Conviértelas a número: reemplaza `,` por `.` y usa `.astype(float)`. Luego muestra la fila de Medellín (`cod_mpio == "05001"`).

In [ ]:
# TU CÓDIGO AQUÍ
for col in ["latitud", "longitud"]:
    divipola[col] = ...

divipola[divipola["cod_mpio"] == "05001"]

**El problema de cruzar fuentes.** Colpensiones escribe `BOGOTA` y el DANE `BOGOTÁ, D.C.`; `CALI` frente a
`SANTIAGO DE CALI`. Para unirlas armamos una **clave**: `departamento|municipio` normalizado (sin tildes ni signos).

**Ejercicio 2.** Cruza por clave **sin alias** y cuenta cuántas filas de `afil_real` no encuentran su municipio
y **cuántos afiliados** suman esas filas.

In [ ]:
# TU CÓDIGO AQUÍ
clave_dane = divipola["dpto"].map(normalizar_texto) + "|" + divipola["nom_mpio"].map(normalizar_texto)
clave_colp = ...
no_cruzan = ...
print("Filas sin cruce:", len(no_cruzan), "· afiliados:", f"{no_cruzan['cantidad'].sum():,}")

¡Bogotá, Cali y Cúcuta no cruzaban! Si no lo revisas, **el 38 % de los afiliados del país desaparece del tablero**.
La solución es una **tabla de alias** revisada a mano, y un **miembro desconocido** (`00000`) para lo que no tiene
municipio (`SIN CIUDAD`): así nada se pierde en silencio.

In [ ]:
ALIAS_MUNICIPIO = {   # (departamento, municipio) como lo escribe Colpensiones → código DIVIPOLA
    ("bogota d c", "bogota"): "11001", ("bolivar", "cartagena"): "13001",
    ("bolivar", "mompos"): "13468", ("cauca", "sotara"): "19760",
    ("cordoba", "purisima"): "23586", ("cundinamarca", "ubate"): "25843",
    ("guainia", "barranco minas"): "94343", ("meta", "fuentedeoro"): "50287",
    ("narino", "cuaspud"): "52224", ("narino", "tumaco"): "52835",
    ("norte de santander", "cucuta"): "54001", ("san andres", "providencia"): "88564",
    ("san andres", "san andres"): "88001", ("sucre", "tolu viejo"): "70823",
    ("tolima", "mariquita"): "73443", ("valle del cauca", "cali"): "76001",
    ("sin ciudad", "sin ciudad"): "00000",      # miembro desconocido
    ("guainia", "mapiripana"): "00000",         # área no municipalizada: no tiene código en DIVIPOLA
}


def codificar_municipio(df, divipola):
    """Agrega cod_mpio (DIVIPOLA) a una tabla de Colpensiones: primero por nombre, luego por alias."""
    clave_dane = divipola["dpto"].map(normalizar_texto) + "|" + divipola["nom_mpio"].map(normalizar_texto)
    por_nombre = dict(zip(clave_dane, divipola["cod_mpio"]))
    claves = list(zip(df["departamento"].map(normalizar_texto), df["municipio"].map(normalizar_texto)))
    df = df.copy()
    df["cod_mpio"] = [por_nombre.get(f"{d}|{m}", ALIAS_MUNICIPIO.get((d, m))) for d, m in claves]
    return df


afil_real = codificar_municipio(afil_real, divipola)
pens_real = codificar_municipio(pens_real, divipola)
print("Sin código después de los alias:", afil_real["cod_mpio"].isna().sum() + pens_real["cod_mpio"].isna().sum())

Con todo cargado medimos la calidad: las 8 reglas de la Sesión 3 más una nueva sobre los datos reales.

In [ ]:
def regla(n, tabla, nombre, dimension, cumple):
    total = len(cumple)
    ok = int(cumple.sum())
    return {"n": n, "tabla": tabla, "regla": nombre, "dimension": dimension,
            "registros": total, "incumplen": total - ok, "pct_cumplimiento": round(ok / total * 100, 2)}

def evaluar_calidad(af, ap, pq, catalogo, afil_real):
    """8 reglas sobre los datos del curso + 1 sobre los datos reales."""
    return pd.DataFrame([
        regla(1, "afiliados", "Documento con 6 a 10 dígitos", "Validez",
              af["numero_documento"].fillna("").str.fullmatch(r"\d{6,10}")),
        regla(2, "afiliados", "Documento único", "Unicidad",
              ~af["numero_documento"].duplicated(keep=False) & af["numero_documento"].notna()),
        regla(3, "afiliados", "Salario >= SMMLV", "Validez", af["salario_base"] >= CONFIG["SMMLV"]),
        regla(4, "afiliados", "Email con formato válido", "Validez",
              af["email"].fillna("").str.fullmatch(r"[\w.+-]+@[\w-]+\.[\w.]+")),
        regla(5, "afiliados", "Ciudad en catálogo", "Consistencia", af["ciudad"].isin(catalogo["ciudad"])),
        regla(6, "aportes", "Aporte = 16 % del IBC (±1 peso)", "Exactitud",
              (ap["valor_aporte"] - ap["ibc"] * CONFIG["TASA_APORTE"]).abs() <= 1),
        regla(7, "aportes", "El afiliado existe", "Integridad", ap["id_afiliado"].isin(af["id_afiliado"])),
        regla(8, "pqrs", "Fechas coherentes", "Consistencia",
              ~((pq["estado"] == "Cerrado") & pq["fecha_respuesta"].isna())
              & ~(pq["fecha_respuesta"] < pq["fecha_radicacion"])),
        regla(9, "reales", "Municipio real con código DIVIPOLA", "Integridad",
              afil_real["cod_mpio"].notna() & (afil_real["cod_mpio"] != "00000")),
    ])


calidad = evaluar_calidad(afiliados, aportes, pqrs, catalogo, afil_real)
calidad

## Parte 3 · Modelo formal: tablas, llaves y reglas
Un **modelo en estrella**: tablas de **dimensiones** (quién, dónde, cuándo) alrededor de tablas de **hechos**
(lo que pasó y se mide).

```
                 dim_departamento
                        │ cod_dpto
                 dim_municipio ───────────── real_afiliados, real_pensionados   (datos reales)
                        │ cod_mpio
 dim_periodo ──── fact_aporte ──── dim_afiliado ──── fact_pqrs
   periodo          (hechos)        id_afiliado        (hechos)
```

| Palabra | Qué significa | En Excel sería |
|---|---|---|
| `PRIMARY KEY` | Identifica cada fila; no se repite ni puede faltar | Una columna de códigos únicos |
| `REFERENCES` (llave foránea) | El valor **debe existir** en otra tabla | Un BUSCARV que nunca da #N/D |
| `NOT NULL` | La columna es obligatoria | Validación de datos: "no permitir vacíos" |
| `CHECK` | Regla que todo valor debe cumplir | Validación de datos con una fórmula |

`dim_afiliado` **no** guarda nombre, documento, correo ni teléfono: el documento va seudonimizado (SHA-256 con sal).
Es lo que pide la Ley 1581 de 2012: guardar solo lo necesario.

In [ ]:
DDL = """CREATE TABLE dim_departamento (
    cod_dpto      TEXT PRIMARY KEY,
    departamento  TEXT NOT NULL
);
CREATE TABLE dim_municipio (
    cod_mpio      TEXT PRIMARY KEY,
    cod_dpto      TEXT NOT NULL REFERENCES dim_departamento (cod_dpto),
    municipio     TEXT NOT NULL,
    latitud       DOUBLE PRECISION,
    longitud      DOUBLE PRECISION
);
CREATE TABLE dim_periodo (
    periodo       TEXT PRIMARY KEY,
    fecha         DATE NOT NULL,
    anio          INTEGER NOT NULL,
    mes           INTEGER NOT NULL CHECK (mes BETWEEN 1 AND 12),
    trimestre     INTEGER NOT NULL
);
CREATE TABLE dim_afiliado (
    id_afiliado   TEXT PRIMARY KEY,
    doc_seudo     TEXT,
    genero        TEXT CHECK (genero IN ('F', 'M')),
    anio_nacimiento INTEGER,
    cod_mpio      TEXT REFERENCES dim_municipio (cod_mpio),
    regional      TEXT,
    estado        TEXT,
    salario_base  BIGINT,
    vigente       INTEGER NOT NULL CHECK (vigente IN (0, 1))
);
CREATE TABLE fact_aporte (
    id_aporte     INTEGER PRIMARY KEY,
    id_afiliado   TEXT NOT NULL REFERENCES dim_afiliado (id_afiliado),
    periodo       TEXT NOT NULL REFERENCES dim_periodo (periodo),
    ibc           BIGINT NOT NULL,
    valor_aporte  BIGINT NOT NULL,
    fecha_pago    DATE,
    operador      TEXT,
    aporte_incorrecto INTEGER NOT NULL
);
CREATE TABLE fact_pqrs (
    radicado      TEXT PRIMARY KEY,
    id_afiliado   TEXT NOT NULL REFERENCES dim_afiliado (id_afiliado),
    tipo          TEXT NOT NULL,
    tema          TEXT NOT NULL,
    canal         TEXT NOT NULL,
    fecha_radicacion DATE NOT NULL,
    fecha_respuesta  DATE,
    estado        TEXT NOT NULL,
    dias_habiles  INTEGER,
    a_tiempo      INTEGER
);
CREATE TABLE real_afiliados (
    cod_mpio      TEXT NOT NULL REFERENCES dim_municipio (cod_mpio),
    genero        TEXT NOT NULL,
    cantidad      INTEGER NOT NULL CHECK (cantidad >= 0),
    PRIMARY KEY (cod_mpio, genero)
);
CREATE TABLE real_pensionados (
    cod_mpio      TEXT PRIMARY KEY REFERENCES dim_municipio (cod_mpio),
    cantidad      INTEGER NOT NULL CHECK (cantidad >= 0)
);
CREATE TABLE real_pensionados_edad (
    rango_edad    TEXT NOT NULL,
    rango_salarial TEXT NOT NULL,
    genero        TEXT NOT NULL,
    tipo_pension  TEXT NOT NULL,
    cantidad      INTEGER NOT NULL CHECK (cantidad >= 0),
    PRIMARY KEY (rango_edad, rango_salarial, genero, tipo_pension)
);
CREATE TABLE calidad_regla (
    n             INTEGER PRIMARY KEY,
    tabla         TEXT NOT NULL,
    regla         TEXT NOT NULL,
    dimension     TEXT NOT NULL,
    registros     INTEGER NOT NULL,
    incumplen     INTEGER NOT NULL,
    pct_cumplimiento DOUBLE PRECISION NOT NULL
);
"""
print(DDL[:900])

In [ ]:
if os.path.exists(RUTA_DB):
    os.remove(RUTA_DB)                         # empezamos de cero cada vez
con = sqlite3.connect(RUTA_DB)
con.execute("PRAGMA foreign_keys = ON")        # SQLite revisa las llaves foráneas solo si se lo pides
con.executescript(DDL)

def sql(consulta):
    """Ejecuta una consulta SQL y devuelve un DataFrame."""
    return pd.read_sql(consulta, con)

sql("SELECT name AS tabla FROM sqlite_master WHERE type = 'table'")

## Parte 4 · Cargar la base
`construir_modelo()` arma cada tabla con **exactamente** las columnas del `CREATE TABLE` y `to_sql()` las inserta,
primero las dimensiones y después los hechos (una llave foránea necesita que el valor ya exista).
Los 72 aportes huérfanos (regla 7) se quedan fuera.

In [ ]:
def seudonimizar_doc(documento):
    """SHA-256 con sal: mismo documento → mismo código, pero no se puede devolver."""
    if pd.isna(documento) or documento == "":
        return None
    return __import__("hashlib").sha256((CONFIG["SAL"] + str(documento)).encode()).hexdigest()[:12]

def construir_modelo(af, ap, pq, catalogo, afil_real, pens_real, edad_real, divipola, calidad):
    """Arma cada tabla del modelo con exactamente las columnas del DDL."""
    # Dimensiones geográficas (DANE) + miembro desconocido
    dim_departamento = (divipola[["cod_dpto", "dpto"]].drop_duplicates()
                        .rename(columns={"dpto": "departamento"}))
    dim_departamento.loc[len(dim_departamento)] = ["00", "SIN INFORMACIÓN"]
    dim_municipio = divipola.rename(columns={"nom_mpio": "municipio"})[
        ["cod_mpio", "cod_dpto", "municipio", "latitud", "longitud"]].copy()
    for c in ["latitud", "longitud"]:
        dim_municipio[c] = pd.to_numeric(dim_municipio[c].astype(str).str.replace(",", "."))  # coma decimal
    dim_municipio.loc[len(dim_municipio)] = ["00000", "00", "SIN CIUDAD", None, None]

    # Periodos
    fechas = pd.to_datetime([f"2026-{m:02d}-01" for m in CONFIG["MESES"]])
    dim_periodo = pd.DataFrame({"periodo": fechas.strftime("%Y-%m"), "fecha": fechas.date,
                                "anio": fechas.year, "mes": fechas.month, "trimestre": fechas.quarter})

    # Afiliados: seudonimizados (sin nombre, documento, email ni teléfono)
    con_doc = af[af["numero_documento"].fillna("") != ""]
    vigentes = set(con_doc.sort_values("id_afiliado").drop_duplicates("numero_documento")["id_afiliado"])
    cat = catalogo.assign(cod_mpio=catalogo["codigo_dane"].astype(str).str.zfill(5))
    dim_afiliado = af.merge(cat[["ciudad", "cod_mpio", "regional"]], on="ciudad", how="left")
    dim_afiliado = pd.DataFrame({
        "id_afiliado": dim_afiliado["id_afiliado"],
        "doc_seudo": dim_afiliado["numero_documento"].map(seudonimizar_doc),
        "genero": dim_afiliado["genero"].where(dim_afiliado["genero"].isin(["F", "M"])),
        "anio_nacimiento": dim_afiliado["fecha_nacimiento"].dt.year.astype("Int64"),
        "cod_mpio": dim_afiliado["cod_mpio"],
        "regional": dim_afiliado["regional"],
        "estado": dim_afiliado["estado"],
        "salario_base": dim_afiliado["salario_base"].astype("Int64"),
        "vigente": dim_afiliado["id_afiliado"].isin(vigentes).astype(int),
    })

    # Hechos: solo aportes de afiliados que existen (los huérfanos quedan fuera: regla 7)
    validos = ap[ap["id_afiliado"].isin(af["id_afiliado"])].reset_index(drop=True)
    fact_aporte = pd.DataFrame({
        "id_aporte": validos.index + 1,
        "id_afiliado": validos["id_afiliado"], "periodo": validos["periodo"],
        "ibc": validos["ibc"], "valor_aporte": validos["valor_aporte"],
        "fecha_pago": validos["fecha_pago"].dt.date, "operador": validos["operador"],
        "aporte_incorrecto": ((validos["valor_aporte"] - validos["ibc"] * CONFIG["TASA_APORTE"]).abs() > 1).astype(int),
    })
    pq = dias_habiles_pqrs(pq)
    fact_pqrs = pq[["radicado", "id_afiliado", "tipo", "tema", "canal", "fecha_radicacion",
                    "fecha_respuesta", "estado", "dias_habiles", "a_tiempo"]].copy()
    for c in ["fecha_radicacion", "fecha_respuesta"]:
        fact_pqrs[c] = fact_pqrs[c].dt.date

    # Datos reales
    real_afiliados = afil_real.groupby(["cod_mpio", "genero"], as_index=False)["cantidad"].sum()
    real_pensionados = pens_real.groupby("cod_mpio", as_index=False)["cantidad"].sum()
    real_pensionados_edad = edad_real.rename(columns={"rango_salarial_smlv": "rango_salarial",
                                                      "cantidad_personas": "cantidad"})[
        ["rango_edad", "rango_salarial", "genero", "tipo_pension", "cantidad"]]

    calidad_regla = calidad.copy()
    return {"dim_departamento": dim_departamento, "dim_municipio": dim_municipio,
            "dim_periodo": dim_periodo, "dim_afiliado": dim_afiliado,
            "fact_aporte": fact_aporte, "fact_pqrs": fact_pqrs,
            "real_afiliados": real_afiliados, "real_pensionados": real_pensionados,
            "real_pensionados_edad": real_pensionados_edad, "calidad_regla": calidad_regla}


tablas = construir_modelo(afiliados, aportes, pqrs, catalogo, afil_real, pens_real, edad_real, divipola, calidad)
for nombre, df in tablas.items():
    df.to_sql(nombre, con, if_exists="append", index=False)
con.commit()
sql("""
SELECT 'dim_afiliado' AS tabla, COUNT(*) AS filas FROM dim_afiliado
UNION ALL SELECT 'fact_aporte', COUNT(*) FROM fact_aporte
UNION ALL SELECT 'fact_pqrs', COUNT(*) FROM fact_pqrs
UNION ALL SELECT 'dim_municipio', COUNT(*) FROM dim_municipio
UNION ALL SELECT 'real_afiliados', COUNT(*) FROM real_afiliados
""")

**La base se defiende sola.** Intentemos meter un aporte de un afiliado que no existe y un género inválido.
En Excel esto entraría sin avisar; aquí la base lo rechaza.

In [ ]:
intentos = {
    "aporte huérfano": "INSERT INTO fact_aporte (id_aporte, id_afiliado, periodo, ibc, valor_aporte, aporte_incorrecto) "
                       "VALUES (999999, 'AF899999', '2026-01', 2000000, 320000, 0)",
    "género inválido": "UPDATE dim_afiliado SET genero = 'X' WHERE id_afiliado = 'AF100026'",
    "mes 13":          "INSERT INTO dim_periodo VALUES ('2026-13', '2026-13-01', 2026, 13, 5)",
}
for nombre, sentencia in intentos.items():
    try:
        con.execute(sentencia)
        print(f"{nombre}: se aceptó (¡ojo!)")
    except sqlite3.IntegrityError as e:
        print(f"{nombre}: RECHAZADO → {e}")

## Parte 5 · Consultas SQL
Una consulta se lee casi en español. El orden de las cláusulas siempre es el mismo:

```sql
SELECT   columnas o cálculos        -- qué quiero ver
FROM     tabla                      -- de dónde
JOIN     otra_tabla ON llave = llave  -- con qué la uno
WHERE    condición sobre filas      -- qué filas entran
GROUP BY columnas                   -- cómo agrupo
HAVING   condición sobre grupos     -- qué grupos quedan
ORDER BY columna DESC               -- cómo ordeno
LIMIT    10                         -- cuántas muestro
```

**Ejemplo resuelto.** Los municipios de Antioquia más al norte (mayor latitud):

In [ ]:
sql("""
SELECT m.cod_mpio, m.municipio, m.latitud
FROM dim_municipio m
WHERE m.cod_dpto = '05'
ORDER BY m.latitud DESC
LIMIT 5
""")

**Ejercicio 3 · `WHERE` y `COUNT`.** ¿Cuántas PQRS del canal **Web** se respondieron **fuera** de término
(`a_tiempo = 0`)?

In [ ]:
# TU CÓDIGO AQUÍ
sql("""
SELECT COUNT(*) AS pqrs_web_tarde
FROM fact_pqrs
WHERE ...
""")

**Ejercicio 4 · `GROUP BY`.** Recaudo total (`SUM(valor_aporte)`) y aportantes únicos
(`COUNT(DISTINCT id_afiliado)`) **por periodo**, ordenado por periodo.

In [ ]:
# TU CÓDIGO AQUÍ
sql("""
SELECT periodo,
       ...
FROM fact_aporte
GROUP BY ...
ORDER BY periodo
""")

**Ejercicio 5 · `JOIN` (datos reales).** Los **10 departamentos** con más afiliados a Colpensiones.
Une `real_afiliados` → `dim_municipio` (por `cod_mpio`) → `dim_departamento` (por `cod_dpto`).

In [ ]:
# TU CÓDIGO AQUÍ
sql("""
SELECT d.departamento, SUM(r.cantidad) AS afiliados
FROM real_afiliados r
JOIN dim_municipio m    ON ...
JOIN dim_departamento d ON ...
GROUP BY d.departamento
ORDER BY afiliados DESC
LIMIT 10
""")

**Ejercicio 6 · `JOIN` + cálculo (datos reales).** Pensionados **por cada 100 afiliados** en cada
departamento con más de 50.000 afiliados. ¿Dónde hay más pensionados en proporción?
Pista: la vista `v_real_departamento` aún no existe; usa dos subconsultas con `WITH` o calcula sobre `real_pensionados`.

In [ ]:
# TU CÓDIGO AQUÍ
sql("""
WITH af AS (
    SELECT m.cod_dpto, SUM(r.cantidad) AS afiliados
    FROM real_afiliados r JOIN dim_municipio m ON m.cod_mpio = r.cod_mpio
    GROUP BY m.cod_dpto
), pe AS (
    ...   -- lo mismo, pero con real_pensionados
)
SELECT d.departamento, af.afiliados, pe.pensionados,
       ROUND(100.0 * pe.pensionados / af.afiliados, 1) AS pensionados_por_100
FROM af
JOIN pe               ON pe.cod_dpto = af.cod_dpto
JOIN dim_departamento d ON d.cod_dpto = af.cod_dpto
WHERE ...
ORDER BY pensionados_por_100 DESC
LIMIT 8
""")

**Ejercicio 7 · `HAVING`.** Temas de PQRS cuyo **% a tiempo** está **por debajo de la meta del 90 %**.
`AVG(a_tiempo)` da la proporción (los vacíos no cuentan); `HAVING` filtra los grupos.

In [ ]:
# TU CÓDIGO AQUÍ
sql("""
SELECT tema,
       COUNT(*)                         AS radicadas,
       ROUND(100.0 * AVG(a_tiempo), 1)  AS pct_a_tiempo
FROM fact_pqrs
GROUP BY tema
HAVING ...
ORDER BY pct_a_tiempo
""")

**Ejercicio 8 · `WITH` (CTE) y `LEFT JOIN`.** La lista de **mora**: afiliados vigentes (`vigente = 1`),
**activos**, con **3 o más** de los 9 meses sin aporte. Cuéntalos **por regional**.
El `LEFT JOIN` es clave: quien no aportó **ningún** mes no aparece en `fact_aporte` y aun así debe contarse.

In [ ]:
# TU CÓDIGO AQUÍ
sql("""
WITH meses AS (
    SELECT id_afiliado, COUNT(DISTINCT periodo) AS meses_con_aporte
    FROM fact_aporte
    GROUP BY id_afiliado
)
SELECT f.regional, COUNT(*) AS afiliados_en_mora
FROM dim_afiliado f
LEFT JOIN meses x ON ...
WHERE f.vigente = 1
  AND ...
GROUP BY f.regional
ORDER BY afiliados_en_mora DESC
""")

**Ejercicio 9 · Función de ventana `LAG`.** Variación % del recaudo frente al **mes anterior**.
¿Qué mes cayó más? `LAG(x) OVER (ORDER BY periodo)` trae el valor de la fila anterior sin perder las filas.

In [ ]:
# TU CÓDIGO AQUÍ
sql("""
WITH mensual AS (
    SELECT periodo, SUM(valor_aporte) AS recaudo
    FROM fact_aporte
    GROUP BY periodo
)
SELECT periodo, recaudo,
       LAG(recaudo) OVER (ORDER BY periodo) AS recaudo_mes_anterior,
       ...  AS variacion_pct
FROM mensual
ORDER BY periodo
""")

**Ejercicio 10 · Nuestro fondo frente al país.** Para cada ciudad de la muestra del curso: afiliados en la muestra
(`dim_afiliado`, solo `vigente = 1`) y afiliados **reales** de Colpensiones en ese mismo municipio. El código DIVIPOLA
es lo que permite unir dos fuentes que nunca se conocieron.

In [ ]:
# TU CÓDIGO AQUÍ
sql("""
SELECT m.municipio,
       COUNT(*) AS afiliados_muestra,
       (SELECT SUM(r.cantidad) FROM real_afiliados r WHERE ...) AS afiliados_reales
FROM dim_afiliado f
JOIN dim_municipio m ON ...
WHERE f.vigente = 1
GROUP BY m.cod_mpio, m.municipio
ORDER BY afiliados_reales DESC
""")

**Interpreta** (para tu presentación):
- ¿Qué departamento tiene más pensionados por cada 100 afiliados? ¿Qué significa para la sostenibilidad?
- ¿Qué regional concentra la mora? ¿Coincide con la que más recauda?

## Parte 6 · Vistas: la capa "oro" que ve la Dirección
Una **vista** es una consulta guardada con nombre. Se consulta como una tabla, pero siempre se calcula con los datos
del momento. El tablero de Grafana **solo lee vistas**: si mañana cambia una regla, se cambia la vista y todos los
gráficos se actualizan.

In [ ]:
VISTAS = """CREATE VIEW v_recaudo_mensual AS
SELECT p.periodo, p.fecha,
       SUM(a.valor_aporte)            AS recaudo,
       COUNT(DISTINCT a.id_afiliado)  AS aportantes,
       ROUND(CAST(100.0 * (SUM(a.valor_aporte) - LAG(SUM(a.valor_aporte)) OVER (ORDER BY p.periodo))
             / LAG(SUM(a.valor_aporte)) OVER (ORDER BY p.periodo) AS NUMERIC), 2) AS variacion_pct
FROM fact_aporte a
JOIN dim_periodo p ON p.periodo = a.periodo
GROUP BY p.periodo, p.fecha;

CREATE VIEW v_recaudo_regional AS
SELECT f.regional,
       SUM(a.valor_aporte)            AS recaudo,
       COUNT(DISTINCT a.id_afiliado)  AS aportantes
FROM fact_aporte a
JOIN dim_afiliado f ON f.id_afiliado = a.id_afiliado
GROUP BY f.regional;

CREATE VIEW v_pqrs_tema AS
SELECT tema,
       COUNT(*)                       AS radicadas,
       COUNT(a_tiempo)                AS cerradas_validas,
       ROUND(CAST(AVG(dias_habiles) AS NUMERIC), 1)            AS dias_promedio,
       ROUND(CAST(100.0 * SUM(a_tiempo) / COUNT(a_tiempo) AS NUMERIC), 1) AS pct_a_tiempo
FROM fact_pqrs
GROUP BY tema;

CREATE VIEW v_pqrs_canal AS
SELECT canal,
       COUNT(*)                       AS radicadas,
       ROUND(CAST(100.0 * SUM(a_tiempo) / COUNT(a_tiempo) AS NUMERIC), 1) AS pct_a_tiempo
FROM fact_pqrs
GROUP BY canal;

CREATE VIEW v_mora AS
WITH meses AS (
    SELECT id_afiliado, COUNT(DISTINCT periodo) AS meses_con_aporte
    FROM fact_aporte
    GROUP BY id_afiliado
)
SELECT f.id_afiliado, f.doc_seudo, m.municipio, f.regional,
       COALESCE(x.meses_con_aporte, 0)                                       AS meses_con_aporte,
       (SELECT COUNT(*) FROM dim_periodo) - COALESCE(x.meses_con_aporte, 0)  AS meses_sin_aporte
FROM dim_afiliado f
LEFT JOIN meses x        ON x.id_afiliado = f.id_afiliado
LEFT JOIN dim_municipio m ON m.cod_mpio = f.cod_mpio
WHERE f.vigente = 1
  AND f.estado = 'Activo'
  AND (SELECT COUNT(*) FROM dim_periodo) - COALESCE(x.meses_con_aporte, 0) >= 3;

CREATE VIEW v_kpi_fondo AS
SELECT (SELECT SUM(valor_aporte) FROM fact_aporte)                          AS recaudo_total,
       (SELECT COUNT(*) FROM fact_pqrs)                                     AS pqrs_radicadas,
       (SELECT ROUND(CAST(100.0 * SUM(a_tiempo) / COUNT(a_tiempo) AS NUMERIC), 1)
          FROM fact_pqrs)                                                   AS pct_pqrs_a_tiempo,
       (SELECT ROUND(CAST(AVG(dias_habiles) AS NUMERIC), 1) FROM fact_pqrs) AS dias_habiles_promedio,
       (SELECT COUNT(*) FROM v_mora)                                        AS activos_en_mora,
       (SELECT MIN(pct_cumplimiento) FROM calidad_regla)                    AS peor_regla_pct;

CREATE VIEW v_real_departamento AS
WITH af AS (
    SELECT m.cod_dpto, SUM(r.cantidad) AS afiliados
    FROM real_afiliados r JOIN dim_municipio m ON m.cod_mpio = r.cod_mpio
    GROUP BY m.cod_dpto
), pe AS (
    SELECT m.cod_dpto, SUM(r.cantidad) AS pensionados
    FROM real_pensionados r JOIN dim_municipio m ON m.cod_mpio = r.cod_mpio
    GROUP BY m.cod_dpto
)
SELECT d.cod_dpto, d.departamento,
       af.afiliados,
       COALESCE(pe.pensionados, 0) AS pensionados,
       ROUND(CAST(100.0 * COALESCE(pe.pensionados, 0) / af.afiliados AS NUMERIC), 1) AS pensionados_por_100_afiliados
FROM dim_departamento d
JOIN af      ON af.cod_dpto = d.cod_dpto
LEFT JOIN pe ON pe.cod_dpto = d.cod_dpto;

CREATE VIEW v_real_municipio AS
SELECT m.cod_mpio, m.municipio, d.departamento, m.latitud, m.longitud,
       (SELECT SUM(cantidad) FROM real_afiliados r WHERE r.cod_mpio = m.cod_mpio)   AS afiliados,
       (SELECT SUM(cantidad) FROM real_pensionados r WHERE r.cod_mpio = m.cod_mpio) AS pensionados
FROM dim_municipio m
JOIN dim_departamento d ON d.cod_dpto = m.cod_dpto
WHERE m.cod_mpio <> '00000';

CREATE VIEW v_comparativo_ciudades AS
SELECT m.municipio, f.regional,
       COUNT(*)                                                                   AS afiliados_muestra_curso,
       (SELECT SUM(cantidad) FROM real_afiliados r WHERE r.cod_mpio = m.cod_mpio) AS afiliados_reales_colpensiones
FROM dim_afiliado f
JOIN dim_municipio m ON m.cod_mpio = f.cod_mpio
WHERE f.vigente = 1
GROUP BY m.cod_mpio, m.municipio, f.regional;
"""
con.executescript(VISTAS)
sql("SELECT name AS vista FROM sqlite_master WHERE type = 'view'")

In [ ]:
sql("SELECT * FROM v_kpi_fondo")

**Ejercicio 11.** Crea tu propia vista `v_pqrs_tipo` con, por cada `tipo` de PQRS: radicadas, días hábiles
promedio y % a tiempo. Luego consúltala.

In [ ]:
# TU CÓDIGO AQUÍ
con.execute("DROP VIEW IF EXISTS v_pqrs_tipo")
con.execute("""
CREATE VIEW v_pqrs_tipo AS
SELECT tipo,
       ...
FROM fact_pqrs
GROUP BY tipo
""")
sql("SELECT * FROM v_pqrs_tipo ORDER BY radicadas DESC")

## Parte 7 · El tablero
En Colab dibujamos un tablero rápido **leyendo solo vistas**. El docente muestra el mismo tablero en **Grafana**,
conectado a PostgreSQL con **las mismas vistas** (carpeta `tablero/` del repositorio).

In [ ]:
kpi = sql("SELECT * FROM v_kpi_fondo").iloc[0]
mensual = sql("SELECT periodo, recaudo / 1e6 AS recaudo FROM v_recaudo_mensual ORDER BY periodo")
regional = sql("SELECT regional, recaudo / 1e6 AS recaudo FROM v_recaudo_regional ORDER BY recaudo")
tema = sql("SELECT tema, pct_a_tiempo FROM v_pqrs_tema ORDER BY pct_a_tiempo DESC")
deptos = sql("SELECT departamento, afiliados FROM v_real_departamento ORDER BY afiliados DESC LIMIT 8")

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(f"Fondo de pensiones · recaudo ${kpi.recaudo_total / 1e6:,.0f} M · PQRS a tiempo {kpi.pct_pqrs_a_tiempo} % · "
             f"mora {int(kpi.activos_en_mora)} afiliados", fontsize=12, fontweight="bold")
ax[0, 0].bar(mensual["periodo"].str[5:], mensual["recaudo"], color="#0B3C91")
ax[0, 0].set_ylim(900, 980); ax[0, 0].set_title("Recaudo mensual (millones de $)")
ax[0, 1].barh(regional["regional"], regional["recaudo"], color="#0B3C91")
ax[0, 1].set_title("Recaudo por regional (millones de $)")
colores = ["#008542" if v >= 90 else "#B02A26" for v in tema["pct_a_tiempo"]]
ax[1, 0].barh(tema["tema"], tema["pct_a_tiempo"], color=colores)
ax[1, 0].axvline(90, color="black", ls="--"); ax[1, 0].set_xlim(80, 95)
ax[1, 0].set_title("% de PQRS a tiempo por tema (meta 90 %)")
ax[1, 1].barh(deptos["departamento"][::-1], deptos["afiliados"][::-1] / 1e6, color="#008542")
ax[1, 1].set_title("Datos reales: afiliados a Colpensiones (millones)")
fig.tight_layout()
fig.savefig("salidas/tablero.png", dpi=110)
plt.show()

**Llévate la base.** El archivo `salidas/bodega_pensiones.db` se abre con DB Browser for SQLite o DBeaver
(gratuitos) y con Power BI o Excel mediante un conector SQLite.

In [ ]:
con.commit()
try:
    from google.colab import files
    files.download(RUTA_DB)
except ImportError:
    print("Base guardada en:", os.path.abspath(RUTA_DB))

### Para la casa: el mismo tablero en Grafana
En el repositorio, `05-proyecto-integrador/tablero/` levanta PostgreSQL + Grafana con Docker:

```bash
cd 05-proyecto-integrador/tablero
docker compose up -d          # luego abre http://localhost:3300
```

`pipeline_bodega.py` es la versión de producción de este notebook: genera la base SQLite y la carga de PostgreSQL.

## Presenta tu hallazgo (3 minutos)
1. Una cifra del fondo y una cifra real del país.
2. La consulta SQL que la respondió.
3. ¿Qué regla de calidad o qué alias te preocupa y qué harías en la fuente?

### Bonus
- Agrega una regla `CHECK (valor_aporte >= 0)` y prueba que la base rechaza un aporte negativo.
- Crea una vista con los pensionados reales por **tipo de pensión** y **género** (`real_pensionados_edad`).
- Lee en vivo un conjunto de datos.gov.co con su API y compáralo con la copia del curso.